# День 8: Pandas — Анализ данных каротажа

**Цель дня:** научиться глубоко анализировать данные скважины — группировать, считать статистику по пластам, находить корреляции и строить информативные графики.

**Итог дня:** ты возьмёшь реальный LAS-файл скважины A15 и сделаешь из него полноценный петрофизический анализ.

---

## Что изучаем сегодня:

| Тема | Зачем геологу |
|------|---------------|
| **Работа с NaN** | В каротаже всегда есть пропуски (-999.25) |
| **Новые столбцы** | Считаем VCL, категории пластов |
| **groupby** | Средние значения по литотипам |
| **pivot_table** | Сводная таблица пласт × параметр |
| **Корреляция** | Связь GR ↔ Porosity ↔ Resistivity |
| **Графики** | Кросс-плоты, гистограммы, каротажный планшет |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 10

print("Pandas:", pd.__version__)
print("NumPy: ", np.__version__)
print("Библиотеки загружены!")


---
## Часть 1: Загружаем реальный LAS-файл скважины A15

Это настоящие данные каротажа: глубина 1808–2084 м, шаг 0.5 м.

Кривые: **GR** — гамма-каротаж, **Perm** — проницаемость (мД),
**Porosity** — пористость (д.е.), **Resistivity** — сопротивление (Ом·м),
**LITH** — код литологии.


In [ ]:
# Загружаем LAS-файл (skiprows=32 — пропускаем заголовок)
df = pd.read_csv("A15", skiprows=32, sep=r"\s+", header=None,
                 names=["DEPTH", "GR", "Perm", "Porosity", "Resistivity", "LITH"])

# Заменяем null-значение LAS (-999.25) на NaN
df = df.replace(-999.25, np.nan)
df["LITH"] = df["LITH"].astype("Int64")

print(f"Загружено строк: {len(df)}")
print(f"Глубины: {df['DEPTH'].min():.1f} — {df['DEPTH'].max():.1f} м")
df.head()


---
## Часть 2: Работа с пропусками (NaN)

**NaN** (Not a Number) — отсутствующее значение. В каротажных данных это норма:
прибор не работал, данные отбракованы, или в LAS стояло -999.25.

| Метод | Что делает |
|-------|------------|
| `df.isnull().sum()` | Считает NaN в каждом столбце |
| `df.dropna()` | Удаляет строки с любым NaN |
| `df.fillna(value)` | Заполняет NaN конкретным значением |
| `df.interpolate()` | Линейная интерполяция по NaN |


In [ ]:
# Смотрим где есть пропуски
print("Количество NaN в каждом столбце:")
print(df.isnull().sum())
print()
print(f"Всего строк: {len(df)}")
print(f"Строк без NaN: {df.dropna().shape[0]}")


In [ ]:
# Карта качества данных — график
cols = ["GR", "Perm", "Porosity", "Resistivity", "LITH"]
null_pct = (df[cols].isnull().sum() / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(cols, null_pct,
               color=["#e74c3c" if v > 5 else "#2ecc71" for v in null_pct])
ax.set_xlabel("% пропущенных значений")
ax.set_title("Качество данных скважины A15")
ax.axvline(x=5, color="orange", linestyle="--", alpha=0.7, label="Порог 5%")
for bar, val in zip(bars, null_pct):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val}%", va="center")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Очищаем данные
df_clean = df.dropna(subset=["GR", "Porosity", "Resistivity", "LITH"]).copy()
df_clean["Perm"] = df_clean["Perm"].interpolate(method="linear")

print(f"Строк до очистки:  {len(df)}")
print(f"Строк после очистки: {len(df_clean)}")
print()
print("NaN после очистки:")
print(df_clean.isnull().sum())


---
## Часть 3: Создание новых столбцов

В петрофизике мы часто вычисляем производные параметры из базовых кривых.
Pandas позволяет создавать новые столбцы прямо из формул — **как Excel, но мощнее**.

```python
df["новый_столбец"] = df["col1"] + df["col2"]   # арифметика
df["новый_столбец"] = df.apply(func, axis=1)    # функция от каждой строки
```

### Формула VCL (объём глин) — Ларионов

```
IGR = (GR - GR_min) / (GR_max - GR_min)         ← нормализованный ГК
VCL = 0.083 × (2^(3.7 × IGR) − 1)               ← формула Ларионова
```


In [ ]:
# Вычисляем VCL (объём глин) по формуле Ларионова
GR_min = df_clean["GR"].quantile(0.05)  # 5-й перцентиль = "чистый" песчаник
GR_max = df_clean["GR"].quantile(0.95)  # 95-й перцентиль = "чистая" глина

IGR = (df_clean["GR"] - GR_min) / (GR_max - GR_min)
IGR = IGR.clip(0, 1)

df_clean["VCL"] = (0.083 * (2 ** (3.7 * IGR) - 1)).clip(0, 1).round(3)

print(f"GR_min (5-й перцентиль):  {GR_min:.1f} API")
print(f"GR_max (95-й перцентиль): {GR_max:.1f} API")
print()
print("Статистика VCL:")
print(df_clean["VCL"].describe().round(3))


In [ ]:
# Категория пласта на основе GR
def kategoriya(row):
    if row["GR"] < 50 and row["Porosity"] > 0.10:
        return "Коллектор"
    elif row["GR"] < 70:
        return "Переходный"
    else:
        return "Глина/Экран"

df_clean["Kategoria"] = df_clean.apply(kategoriya, axis=1)

print("Распределение по категориям:")
print(df_clean["Kategoria"].value_counts())
print()
df_clean[["DEPTH", "GR", "Porosity", "VCL", "Kategoria"]].head(10)


### Задание 1
Создай новый столбец `Log_Perm` — логарифм (base 10) проницаемости: `np.log10(df_clean["Perm"])`

*Почему логарифм?* Проницаемость меняется на несколько порядков (0.001–1000 мД),
логарифм делает распределение более симметричным — это важно для ML.


In [ ]:
# Твой код здесь:


---
## Часть 4: groupby — статистика по литотипам

`groupby` разбивает таблицу на группы и считает агрегат для каждой.

```python
df.groupby("столбец")["данные"].mean()   # среднее по группам
df.groupby("столбец").agg({...})          # несколько метрик сразу
```

**Аналогия:** в Excel это сводная таблица. В SQL — GROUP BY.

В нашем файле LITH кодирует тип породы цифрами:

| Код | Порода |
|-----|--------|
| 1   | Песчаник |
| 2   | Карбонат |
| 3   | Глина |


In [ ]:
# Добавляем читаемые названия литологии
lith_names = {1: "Песчаник", 2: "Карбонат", 3: "Глина"}
df_clean["LITH_name"] = df_clean["LITH"].map(lith_names).fillna("Другое")

# Среднее ГК по типам пород
print("Среднее GR по литотипам:")
print(df_clean.groupby("LITH_name")["GR"].mean().round(1))

print("\nКоличество замеров по типам пород:")
print(df_clean.groupby("LITH_name").size())


In [ ]:
# Несколько метрик сразу через agg()
stats = df_clean.groupby("LITH_name").agg(
    GR_mean    = ("GR",          "mean"),
    GR_std     = ("GR",          "std"),
    Por_mean   = ("Porosity",    "mean"),
    Res_median = ("Resistivity", "median"),
    VCL_mean   = ("VCL",         "mean"),
    Count      = ("DEPTH",       "count")
).round(3)

print("Сводная статистика по литотипам:")
stats


In [ ]:
# Визуализация — сравнение параметров по литотипам
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Средние петрофизические параметры по литотипам (скв. A15)",
             fontsize=12, fontweight="bold")

colors = {"Песчаник": "#f39c12", "Карбонат": "#3498db",
          "Глина": "#95a5a6", "Другое": "#bdc3c7"}
lith_order = [l for l in ["Песчаник", "Карбонат", "Глина", "Другое"]
              if l in stats.index]

for ax, col, title, ylabel in zip(
        axes,
        ["GR_mean", "Por_mean", "Count"],
        ["Среднее GR (API)", "Средняя пористость", "Количество замеров"],
        ["GR, API", "Porosity, д.е.", "Точек"]):
    vals = stats.loc[lith_order, col]
    bars = ax.bar(lith_order, vals,
                  color=[colors[l] for l in lith_order],
                  edgecolor="k", linewidth=0.5)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01,
                f"{val:.3g}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


### Задание 2
Используя `groupby`, найди **максимальную проницаемость** `Perm` для каждого литотипа.
В каком типе пород она выше? Почему это логично с геологической точки зрения?


In [ ]:
# Твой код здесь:


---
## Часть 5: pivot_table — сводная таблица

`pivot_table` — это `groupby` на стероидах.
Позволяет группировать по **двум осям** одновременно.

```python
pd.pivot_table(df,
    values  = "что_считаем",
    index   = "строки",
    columns = "столбцы",   # необязательно
    aggfunc = "как_считаем"
)
```

**Аналогия:** в Excel это именно «Сводная таблица» (PivotTable).


In [ ]:
# Создаём зоны по глубине (~50 м каждая)
df_clean["Zone"] = pd.cut(
    df_clean["DEPTH"], bins=6,
    labels=["1808-1854","1854-1900","1900-1946",
            "1946-1992","1992-2038","2038-2084"])

# Сводная таблица: зона × литотип → среднее GR
pivot = pd.pivot_table(df_clean,
                       values="GR",
                       index="Zone",
                       columns="LITH_name",
                       aggfunc="mean").round(1)

print("Среднее GR по зонам и литотипам:")
pivot


In [ ]:
# Тепловая карта — очень наглядно!
fig, ax = plt.subplots(figsize=(8, 4))

pivot_plot = pivot.fillna(pivot.mean())
im = ax.imshow(pivot_plot.values, aspect="auto", cmap="RdYlGn_r")
plt.colorbar(im, ax=ax, label="Среднее GR (API)")

ax.set_xticks(range(len(pivot_plot.columns)))
ax.set_xticklabels(pivot_plot.columns, rotation=15)
ax.set_yticks(range(len(pivot_plot.index)))
ax.set_yticklabels(pivot_plot.index)
ax.set_title("Тепловая карта: GR по зонам и литотипам\n"
             "(зелёный = чистый, красный = глинистый)",
             fontweight="bold")

for i in range(len(pivot_plot.index)):
    for j in range(len(pivot_plot.columns)):
        val = pivot_plot.iloc[i, j]
        ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                fontsize=9, color="white", fontweight="bold")

plt.tight_layout()
plt.show()


---
## Часть 6: Корреляция между кривыми

**Корреляция** — насколько два параметра изменяются вместе.

| Значение | Интерпретация |
|----------|---------------|
| +1.0 | Идеальная прямая корреляция |
| 0 | Нет связи |
| −1.0 | Идеальная обратная корреляция |

В каротаже ожидаем:
- **GR ↑ → Porosity ↓** (глины снижают пористость) → **отрицательная**
- **Porosity ↑ → Resistivity ↓** (вода в порах проводит ток) → **отрицательная**


In [ ]:
# Корреляционная матрица
corr_cols = ["GR", "Porosity", "Resistivity", "VCL", "Perm"]
corr = df_clean[corr_cols].corr().round(2)

print("Корреляционная матрица:")
corr


In [ ]:
# Тепловая карта корреляций
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Коэффициент корреляции Пирсона")

ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
ax.set_title("Корреляция каротажных кривых\nСкважина A15",
             fontweight="bold")

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr.values[i, j]
        clr = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=10, color=clr, fontweight="bold")

plt.tight_layout()
plt.show()

print("Синий = положительная | Красный = отрицательная | Белый = слабая связь")


In [ ]:
# Кросс-плоты — классика петрофизики
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Кросс-плоты каротажных кривых (скв. A15)",
             fontsize=13, fontweight="bold")

lith_colors = {"Песчаник": "#f39c12", "Карбонат": "#3498db",
               "Глина": "#7f8c8d", "Другое": "#bdc3c7"}

# Кросс-плот 1: GR vs Porosity
for lith, group in df_clean.groupby("LITH_name"):
    axes[0].scatter(group["GR"], group["Porosity"],
                   c=lith_colors.get(lith, "#bdc3c7"),
                   label=lith, alpha=0.4, s=8, edgecolors="none")
axes[0].set_xlabel("GR (API)")
axes[0].set_ylabel("Porosity (д.е.)")
axes[0].set_title("GR vs Porosity")
axes[0].legend(markerscale=3)
axes[0].grid(True, alpha=0.3)

# Кросс-плот 2: Porosity vs Resistivity (лог. шкала)
for lith, group in df_clean.groupby("LITH_name"):
    axes[1].scatter(group["Porosity"], group["Resistivity"],
                   c=lith_colors.get(lith, "#bdc3c7"),
                   label=lith, alpha=0.4, s=8, edgecolors="none")
axes[1].set_xlabel("Porosity (д.е.)")
axes[1].set_ylabel("Resistivity (Ом·м)")
axes[1].set_yscale("log")
axes[1].set_title("Porosity vs Resistivity (лог. шкала)")
axes[1].legend(markerscale=3)
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


---
## Часть 7: Каротажный планшет

Настоящие геологи смотрят на данные вертикально — кривые идут **сверху вниз по глубине**.
Создадим настоящий каротажный планшет для скважины A15!


In [ ]:
# Берём первые 200 точек (100 метров)
df_plot = df_clean.iloc[:200].copy()
depth = df_plot["DEPTH"]

fig, axes = plt.subplots(1, 4, figsize=(12, 10), sharey=True)
fig.suptitle("Каротажный планшет — Скважина A15 (1808–1908 м)",
             fontsize=13, fontweight="bold", y=1.01)

# Трек 1: GR
axes[0].plot(df_plot["GR"], depth, color="green", linewidth=0.8)
axes[0].fill_betweenx(depth, df_plot["GR"], 0, alpha=0.2, color="green")
axes[0].set_xlabel("GR (API)", color="green")
axes[0].set_title("GR", fontweight="bold")
axes[0].invert_yaxis()
axes[0].set_ylabel("Глубина (м)")
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis="x", colors="green")

# Трек 2: VCL
axes[1].plot(df_plot["VCL"], depth, color="brown", linewidth=0.8)
axes[1].fill_betweenx(depth, df_plot["VCL"], 0, alpha=0.3, color="brown")
axes[1].set_xlabel("VCL (д.е.)", color="brown")
axes[1].set_title("VCL (Ларионов)", fontweight="bold")
axes[1].set_xlim(0, 1)
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis="x", colors="brown")

# Трек 3: Porosity
axes[2].plot(df_plot["Porosity"], depth, color="blue", linewidth=0.8)
axes[2].fill_betweenx(depth, df_plot["Porosity"], 0, alpha=0.2, color="blue")
axes[2].set_xlabel("Porosity (д.е.)", color="blue")
axes[2].set_title("NPHI", fontweight="bold")
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(axis="x", colors="blue")

# Трек 4: Resistivity (лог. шкала)
axes[3].semilogx(df_plot["Resistivity"], depth, color="red", linewidth=0.8)
axes[3].set_xlabel("Rt (Ом·м)", color="red")
axes[3].set_title("RT (log)", fontweight="bold")
axes[3].grid(True, which="both", alpha=0.3)
axes[3].tick_params(axis="x", colors="red")

plt.tight_layout()
plt.show()


---
## Часть 8: Сортировка и поиск лучших пластов

```python
df.sort_values("col")                   # по возрастанию
df.sort_values("col", ascending=False)  # по убыванию
df.nlargest(10, "col")                  # топ-10 максимальных
df.nsmallest(10, "col")                 # топ-10 минимальных
```


In [ ]:
# Топ-10 самых пористых интервалов
top_porous = df_clean.nlargest(10, "Porosity")[
    ["DEPTH", "GR", "Porosity", "Resistivity", "VCL", "LITH_name"]]

print("Топ-10 самых пористых интервалов:")
top_porous.round(4)


In [ ]:
# Комплексный рейтинг коллектора
# (низкий GR) + (высокая пористость) + (высокое сопротивление)
df_temp = df_clean.copy()

gr_score  = 1 - (df_temp["GR"] - df_temp["GR"].min()) / \
                (df_temp["GR"].max() - df_temp["GR"].min())
por_score = (df_temp["Porosity"] - df_temp["Porosity"].min()) / \
            (df_temp["Porosity"].max() - df_temp["Porosity"].min())
res_score = (np.log10(df_temp["Resistivity"]) - \
             np.log10(df_temp["Resistivity"].min())) / \
            (np.log10(df_temp["Resistivity"].max()) - \
             np.log10(df_temp["Resistivity"].min()))

# Взвешенный балл (GR 30%, Poros 40%, Res 30%)
df_temp["Score"] = (0.3 * gr_score + 0.4 * por_score + 0.3 * res_score).round(3)

print("ТОП-10 потенциальных коллекторов:")
df_temp.nlargest(10, "Score")[
    ["DEPTH", "GR", "Porosity", "Resistivity", "VCL", "LITH_name", "Score"]
].round(4)


In [ ]:
# Визуализация рейтинга по глубине
fig, axes = plt.subplots(1, 2, figsize=(10, 9), sharey=True)
fig.suptitle("Рейтинг коллекторских свойств — Скважина A15",
             fontsize=12, fontweight="bold")

# GR по глубине
axes[0].plot(df_temp["GR"], df_temp["DEPTH"], color="green",
             linewidth=0.6, alpha=0.8)
axes[0].set_xlabel("GR (API)")
axes[0].set_ylabel("Глубина (м)")
axes[0].set_title("Гамма-каротаж")
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=60, color="orange", linestyle="--",
               alpha=0.7, label="GR = 60 API")
axes[0].legend(fontsize=8)

# Балл коллектора по глубине
sc = axes[1].scatter(df_temp["Score"], df_temp["DEPTH"],
                     c=df_temp["Score"], cmap="RdYlGn",
                     s=5, alpha=0.6, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label="Балл коллектора")
axes[1].set_xlabel("Collector Score")
axes[1].set_title("Рейтинг коллектора")
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0.7, color="green", linestyle="--",
               alpha=0.7, label="Порог 0.7")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

n_good = (df_temp["Score"] >= 0.7).sum()
print(f"Интервалов с баллом ≥ 0.7: {n_good} ({n_good * 0.5:.1f} метров)")


---
## Итоговое задание — Отчёт петрофизика

Ты ведущий петрофизик. Подготовь краткий анализ скважины A15:

1. Выведи **общее количество метров** по каждому литотипу
   (подсказка: `groupby` + `size()` × шаг 0.5 м)

2. Найди **среднюю пористость и сопротивление** только для коллекторов
   (строки где `Score >= 0.7`)

3. Выведи **глубину самого перспективного интервала**
   (максимальный `Score`)


In [ ]:
# Твой код здесь:

# 1. Метры по литотипам


# 2. Среднее по коллекторам


# 3. Лучший пласт



---
## Шпаргалка Дня 8

```python
import pandas as pd
import numpy as np

# === Работа с NaN ===
df.isnull().sum()               # сколько NaN в каждом столбце
df.dropna()                     # удалить строки с любым NaN
df.dropna(subset=["col"])       # удалить только если NaN в col
df["col"].fillna(0)             # заполнить нулём
df["col"].interpolate()         # линейная интерполяция

# === Новые столбцы ===
df["new"] = df["a"] + df["b"]           # арифметика
df["new"] = df.apply(func, axis=1)      # функция от строки
df["cat"] = pd.cut(df["col"], bins=5)   # разбивка на интервалы

# === groupby ===
df.groupby("col")["val"].mean()         # среднее по группам
df.groupby("col").agg({"a": "mean", "b": "max"})
df.groupby("col").size()                # количество в группе

# === pivot_table ===
pd.pivot_table(df, values="v", index="row", columns="col", aggfunc="mean")

# === Корреляция ===
df.corr()                       # матрица корреляций
df["a"].corr(df["b"])           # корреляция двух столбцов

# === Сортировка ===
df.sort_values("col")           # по возрастанию
df.sort_values("col", ascending=False)
df.nlargest(10, "col")          # топ-10 максимальных
df.nsmallest(10, "col")         # топ-10 минимальных
```


---
## Итог Дня 8

Сегодня ты научилась:

- **Обрабатывать NaN** — реальные данные всегда с пропусками
- **Создавать новые столбцы** — VCL, категории, логарифмы прямо в таблице
- **groupby** — статистика по литотипам одной строкой кода
- **pivot_table** — сводная таблица «зона × литотип»
- **Корреляционная матрица** — видеть связи между кривыми каротажа
- **Каротажный планшет** — строить профессиональный вертикальный разрез
- **Рейтинг коллекторов** — находить лучшие пласты по комплексному критерию

**Следующий шаг → День 9: Очистка данных**
Дубликаты, выбросы, несогласованные форматы — работа с "грязными" датасетами.
